# GA vs random winner full-data confirmation

Bu notebook proxy search kazananlarını bütün 91.000 train ve 19.500 validation görüntüsünde eşit koşullarla doğrular. Test satırı manifestte yoktur, test DataLoader'ı kurulmaz ve `test_metrics.json` üretilemez. Bu çalışma final test sonucu değildir. İlk üç kod hücresini çalıştırıp çıktıyı Codex ile incelemeden uzun hücreyi açma.

In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, time, zipfile

assert 'google.colab' in sys.modules, 'Bu notebook Google Colab için hazırlanmıştır.'
from google.colab import drive, files
drive.mount('/content/drive')
print('Drive bağlandı.')

## 1. Repo, veri ve dondurulmuş girdiler
Dataset ZIP geçici Colab diskine çıkarılır. İki adayın checkpointleri ve sonuçları Drive'da ayrı klasörlerde korunur.

In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
ZIP_PATH = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer.zip')
LOCAL_DATA_BASE = Path('/content/NNGA_DATA')
EXTRACT_MARKER = LOCAL_DATA_BASE / '.multi_cancer_extract_complete'
INIT_CHECKPOINT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_26leaf_baseline_v1/best_model.pth')
SOURCE_SEARCH_ROOT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_resnet18_lastblock_search_v1')
SOURCE_SEARCH_SUMMARY = SOURCE_SEARCH_ROOT / 'search_summary.json'
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_full_confirmation_v1')
TOP_LOG = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_full_confirmation_v1_console.log')
REVIEW_ZIP = Path('/content/drive/MyDrive/NNGA_OUTPUTS/ga_random_full_confirmation_v1_review_package.zip')
FROZEN_SPLIT_SHA256 = '999bf7a9bd1bba9f2957a9db0e13ae72adb3ecb8fdbc1f33cce40b083c1cf171'
FROZEN_CONFIRMATION_SHA256 = 'afe9d336d53ae24490e065c37b5cc764d0cfeff8f01b3fb7ee8af756bcd375f9'
FROZEN_INIT_SHA256 = '20613069067825ea0dfa30e2242678b863cd0acd7b9be3c5b88f265d39285177'
FROZEN_SOURCE_SUMMARY_SHA256 = 'debb373153f0c3264e599e6c91a00c1689a86d2a21730cee2bba6d70f4a36f9c'
FROZEN_TRAINER_SHA256 = '807a23e40ea3aaa5c0c98737beddb637d71c5e5a9e5f230aa0b75d0be7212a77'
FROZEN_RUNNER_SHA256 = '69d045c449245b1e408b33017737691c153c25586461d3cf5c8691c2a449fc49'
FROZEN_CONFIG_SHA256 = '7f0886527f24365373842081561a0a4b6318f16fb59d0272fff1266a9bbd34a9'
FROZEN_REPORT_SHA256 = 'eba3d8b626cc8b2ba2f513f4ec58061a28470f9d99b5a095d62461ca5e3d4e6d'
EXPECTED_GROUPS = ('ALL', 'Brain Cancer', 'Breast Cancer', 'Cervical Cancer', 'Kidney Cancer', 'Lung and Colon Cancer', 'Lymphoma', 'Oral Cancer')
EXPECTED_CLASS_NAMES = ('all_benign', 'all_early', 'all_pre', 'all_pro', 'brain_glioma', 'brain_menin', 'brain_tumor', 'breast_benign', 'breast_malignant', 'cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi', 'colon_aca', 'colon_bnt', 'kidney_normal', 'kidney_tumor', 'lung_aca', 'lung_bnt', 'lung_scc', 'lymph_cll', 'lymph_fl', 'lymph_mcl', 'oral_normal', 'oral_scc')

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
SPLIT_MANIFEST = PROJECT_ROOT / 'research/splits/practical_leakage_reduced_split_v1.csv'
CONFIRMATION_MANIFEST = PROJECT_ROOT / 'research/splits/ga_full_confirmation_v1.csv'
CONFIRMATION_REPORT = PROJECT_ROOT / 'research/splits/ga_full_confirmation_v1_report.json'
CONFIG_PATH = PROJECT_ROOT / 'config/ga_resnet18_full_confirmation_v1.json'
RUNNER_PATH = PROJECT_ROOT / 'src/run_ga_random_full_confirmation.py'
TRAINER_PATH = PROJECT_ROOT / 'src/train.py'
for path, expected in ((SPLIT_MANIFEST, FROZEN_SPLIT_SHA256), (CONFIRMATION_MANIFEST, FROZEN_CONFIRMATION_SHA256), (CONFIRMATION_REPORT, FROZEN_REPORT_SHA256), (INIT_CHECKPOINT, FROZEN_INIT_SHA256), (SOURCE_SEARCH_SUMMARY, FROZEN_SOURCE_SUMMARY_SHA256), (TRAINER_PATH, FROZEN_TRAINER_SHA256), (RUNNER_PATH, FROZEN_RUNNER_SHA256), (CONFIG_PATH, FROZEN_CONFIG_SHA256)):
    assert path.is_file(), f'Eksik frozen girdi: {path}'
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected, f'Hash uyuşmuyor: {path} expected={expected} actual={actual}'
assert ZIP_PATH.is_file(), f'Dataset ZIP bulunamadı: {ZIP_PATH}'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert free_gib >= 12.0, 'Dataset açma ve confirmation için en az 12 GiB yerel boş alan gerekli.'
if not EXTRACT_MARKER.is_file():
    LOCAL_DATA_BASE.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print('ZIP Colab diskine çıkarılıyor; birkaç dakika çıktı görünmeyebilir...')
    subprocess.run(['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(LOCAL_DATA_BASE)], check=True)
    EXTRACT_MARKER.write_text('complete\n', encoding='utf-8')
    print(f'ZIP çıkarma tamamlandı: {(time.monotonic() - started) / 60:.1f} dakika')
dataset_candidates = (LOCAL_DATA_BASE / 'Multi Cancer' / 'Multi Cancer', LOCAL_DATA_BASE / 'Multi Cancer', LOCAL_DATA_BASE)
DATASET_ROOT = next((p for p in dataset_candidates if all((p / g).is_dir() for g in EXPECTED_GROUPS)), None)
assert DATASET_ROOT is not None, 'Sekiz ana klasörü içeren dataset kökü bulunamadı.'
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('Git commit:', commit)
print('Dataset:', DATASET_ROOT)
print('Tam train/validation manifest SHA-256:', FROZEN_CONFIRMATION_SHA256)
print('Kaynak search summary SHA-256:', FROZEN_SOURCE_SUMMARY_SHA256)
print('Confirmation output:', OUTPUT_DIR)

## 2. GPU ve validation-only sözleşme preflight
Bu hücre model eğitmez. İki kazananı, 91.000/19.500/0 sayımını, test yasağını ve yalnız `layer4 + fc` kapsamını doğrular.

In [ ]:
import torch, torchvision
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train import build_model, validate_initial_checkpoint
from run_ga_random_full_confirmation import validate_config, validate_source_search
assert torch.cuda.is_available(), 'GPU seçilmemiş; T4 runtime aç.'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
validate_config(config)
source_summary = json.loads(SOURCE_SEARCH_SUMMARY.read_text(encoding='utf-8'))
validate_source_search(source_summary, config)
manifest_report = json.loads(CONFIRMATION_REPORT.read_text(encoding='utf-8'))
assert manifest_report['counts'] == {'train': 91000, 'val': 19500, 'test': 0}
assert manifest_report['leaf_count'] == 26
assert all(counts == {'train': 3500, 'val': 750} for counts in manifest_report['per_leaf'].values())
assert len(config['candidates']) == 2 and {row['method'] for row in config['candidates']} == {'ga', 'random'}
assert {row['candidate_id'] for row in config['candidates']} == {'GA_G01_C02', 'RANDOM_002'}
assert config['test_policy']['test_opened_during_confirmation'] is False
assert config['test_policy']['test_loader_constructed'] is False
assert config['test_policy']['test_metrics_allowed'] is False
checkpoint, checked_hash = validate_initial_checkpoint(INIT_CHECKPOINT, FROZEN_INIT_SHA256, 'resnet18', 'subclass', EXPECTED_CLASS_NAMES)
probe = build_model('resnet18', 26, pretrained=True, fine_tune=True, dropout=0.30)
probe.load_state_dict(checkpoint['model_state_dict'], strict=True)
trainable = {name: p.numel() for name, p in probe.named_parameters() if p.requires_grad}
assert trainable and all(name.startswith(('layer4.', 'fc.')) for name in trainable)
assert sum(trainable.values()) == 8407066
del probe, checkpoint
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Trainable scope: yalnız layer4 + fc | parameter:', sum(trainable.values()))
print('Full confirmation: 91.000 train / 19.500 validation / 0 test')
print('Adaylar:', [(row['method'], row['candidate_id']) for row in config['candidates']])
print('Her aday: en fazla 12 epoch, patience 4, validation macro-F1 checkpoint seçimi')
print('Preflight başarılı. Henüz eğitim başlamadı.')

## 3. Kilitli tam-veri confirmation
İlk üç kod hücresinin çıktısını Codex ile incelemeden aşağıdaki değeri değiştirme. İki aday sırayla çalışır. Runtime kesilirse aynı hücre yeniden çalıştırılır; tamamlanmış aday atlanır ve yarım aday checkpoint'ten devam eder. Output klasörünü silme.

In [ ]:
AUTHORIZE_FULL_CONFIRMATION = False
assert AUTHORIZE_FULL_CONFIRMATION, 'Tam-veri confirmation kilitli; preflight Codex ile onaylanmadan çalışmaz.'
summary_path = OUTPUT_DIR / 'confirmation_summary.json'
if summary_path.is_file():
    existing_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    assert existing_summary.get('status') != 'ga_random_full_confirmation_v1_complete', 'Confirmation zaten tamamlanmış; yeniden çalıştırma.'
command = [
    sys.executable, '-u', str(RUNNER_PATH),
    '--data-dir', str(DATASET_ROOT),
    '--split-manifest', str(SPLIT_MANIFEST),
    '--confirmation-manifest', str(CONFIRMATION_MANIFEST),
    '--init-checkpoint', str(INIT_CHECKPOINT),
    '--source-search-summary', str(SOURCE_SEARCH_SUMMARY),
    '--output-dir', str(OUTPUT_DIR),
    '--config', str(CONFIG_PATH),
    '--workers', '2',
    '--authorize', 'FROZEN_GA_RANDOM_FULL_CONFIRMATION_V1',
]
log_mode = 'a' if TOP_LOG.exists() else 'w'
print('Confirmation komutu:', ' '.join(command))
with TOP_LOG.open(log_mode, encoding='utf-8') as log_handle:
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line); log_handle.flush()
    return_code = process.wait()
print('RETURN CODE:', return_code)
if return_code != 0:
    raise RuntimeError('Confirmation tamamlanamadı. Output klasörünü silme; logu Codex’e gönder.')

## 4. Tamamlanmış validation-only paketi doğrula ve indir
Bu hücre yalnız iki aday tamamlandıktan sonra çalışır. Büyük checkpointler Drive'da kalır; hafif kanıt paketi indirilir.

In [ ]:
required_root = ['run_contract.json', 'frozen_config.json', 'confirmation_results.csv', 'confirmation_summary.json']
missing = [name for name in required_root if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Eksik confirmation çıktıları: {missing}'
summary = json.loads((OUTPUT_DIR / 'confirmation_summary.json').read_text(encoding='utf-8'))
assert summary['status'] == 'ga_random_full_confirmation_v1_complete'
assert summary['completed_candidates'] == 2 and summary['expected_candidates'] == 2
assert summary['test_evaluated'] is False
with (OUTPUT_DIR / 'confirmation_results.csv').open(encoding='utf-8', newline='') as handle:
    result_rows = list(csv.DictReader(handle))
assert len(result_rows) == 2 and {row['method'] for row in result_rows} == {'ga', 'random'}
assert all(row['test_evaluated'].lower() == 'false' for row in result_rows)
assert not list((OUTPUT_DIR / 'candidates').glob('*/test_metrics.json')), 'Confirmation test metriği üretmiş.'
review_paths = [OUTPUT_DIR / name for name in required_root]
if TOP_LOG.is_file(): review_paths.append(TOP_LOG)
for candidate_dir in sorted((OUTPUT_DIR / 'candidates').iterdir()):
    for name in ('candidate.json', 'confirmation_result.json', 'search_trial_summary.json', 'history.csv', 'run_config.json', 'learning_curves.png'):
        path = candidate_dir / name
        assert path.is_file(), f'Eksik confirmation kanıtı: {path}'
        review_paths.append(path)
artifact_hashes = {}
for path in review_paths:
    arcname = path.name if path == TOP_LOG else path.relative_to(OUTPUT_DIR).as_posix()
    artifact_hashes[arcname] = {'sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'size_bytes': path.stat().st_size}
manifest_path = OUTPUT_DIR / 'review_artifact_hashes.json'
manifest_path.write_text(json.dumps(artifact_hashes, indent=2), encoding='utf-8')
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in review_paths:
        arcname = path.name if path == TOP_LOG else path.relative_to(OUTPUT_DIR).as_posix()
        archive.write(path, arcname=arcname)
    archive.write(manifest_path, arcname='review_artifact_hashes.json')
review_hash = hashlib.sha256(REVIEW_ZIP.read_bytes()).hexdigest()
print('Durum:', summary['status'])
print('GA full-validation:', summary['results']['ga'])
print('Random full-validation:', summary['results']['random'])
print('Validation kazananı:', summary['validation_winner'])
print('Test evaluated:', summary['test_evaluated'])
print('Review package:', REVIEW_ZIP)
print('Review package SHA-256:', review_hash)
files.download(str(REVIEW_ZIP))